In [1]:
import numpy as np
import qiskit as qk
import qiskit.circuit.library as ql
import math
from qiskit_aer import AerSimulator
from qiskit import transpile
import random
from fractions import Fraction

# Shor's Implementation

In this notebook we implement Shor's algorithm for factorization. Fundamentally, what we're working to compute is:

\begin{equation*}
    g = \text{GCD}(N, a^{r/2} \pm 1)
\end{equation*}

Where $N$ is the integer we are working to factor, $a$ is a random integer that is co-prime to $N$, and $r$ is the order of $a \mod N$. If $g$ is not $1$, then we've found one factor and the other is simply $N / g$.

The typical steps for Shor's algorithm are given by:

1. For a given integer $N$, pick a random integer $1 < a < N$ which is co-prime to $N$ (that is, $\text{GCD}(a, N) == 1$) and confirm $a$ is not also a factor of $N$, otherwise we have trivially found the factors
2. Utilize a quantum circuit to find the order $r$ of $a \mod N$ using quantum phase estimation
3. If $r$ is not even or $a^{r/2} \equiv -1$, try again. Otherwise, calculate $a^{r/2} \pm 1$ and check if either share a non-trivial divisor with $N$. If they do, we've found our factors. Otherwise we try again

Clearly step 2 is the heart of this implementation, so we'll break that calculation down into several steps below and provide context around each of the functions that implement a portion of this step. The standard steps for calculating $r$ are:

1. Initialize quantum and classical registers. We need a "work" register (initialized to $\ket{1}$), the phase register (initialized to a uniform superposition using $H$), ancilla qubits, and a flag qubit. We additionally need a classical register to store the measurement of our phase register. In addition, we have an accumulator "acc" quantum register that we use to support in-place modular multiplication
2. Perform controlled modular exponentiation: for each phase qubit, $i$, we apply multiplication by $a^{2^i} \mod N$ in our work register
3. Extract phase in the computational basis via an inverse QFT
4. Measure the phase
5. Use the continued fractions algorithm to extract $r$ from the measured qubits

This implementation is based on David Finder's implementation from the Quantum Computing course, which can be found [on GitHub](https://github.com/dfinder/qmod_multiply/blob/main/quantum.py).

### QFT-based modular arithmetic
To begin, we'll define the functions needed for modular exponentiation in our quantum circuit. The intention here is to take a state $\ket{x}$ and transform it:

\begin{equation*}
    \ket{x} \rightarrow \ket{kx \mod N}
\end{equation*}

This is needed for the controlled modular exponentiation. We've opted to implement a quantum Fourier constant adder to support this, which we based on the implementation described in the paper [Efficient Quantum Modular Arithmetics for the ISQ Era](https://arxiv.org/pdf/2311.08555) by Atchade-Adelomou et al. This works by taking our work state $\ket{x}$ in the Fourier basis and applying controlled phase shifts to transform it to $\ket{x + k}$ when converted back to the computational basis via $\text{QFT}^\dagger$.

From this adder we build modular multiplication by exploiting the binary decomposition of $x$: since $kx = \sum_i x_i \cdot k \cdot 2^i$, each bit of $x$ controls a modular addition of the appropriately shifted constant into an accumulator register.

In [2]:
def add_k_fourier(k: int, N: int) -> qk.QuantumCircuit:
    """
    This circuit assumes the target register is already in the Fourier basis.

    If the register encodes some integer x in the Fourier basis, then this
    circuit applies phases so that, after an inverse QFT, the register
    corresponds to |x + k>.
    """
    n = N.bit_length()

    add = qk.QuantumRegister(n + 1, "add")
    qc = qk.QuantumCircuit(add, name=f"add_{k}_fourier")

    # The angle on each qubit is proportional to k / 2^m.
    for j in range(n + 1):
        m = n + 1 - j
        angle = 2 * np.pi * k / (2 ** m)
        qc.p(angle, add[j])

    return qc

In [3]:
def phase_adder(k: int, N: int) -> qk.QuantumCircuit:
    """
    This function updates the circuit to modify the work register with the following transform:
    
        |x>  ->  |x + k mod N>
    
    Implements the adder using the Fourier adder implemented above. The steps:
      1) Add k
      2) Subtract N
      3) Inspect overflow / sign information
      4) If needed, add N back
      5) Uncompute the ancilla flag
    
    The arithmetic additions happen in the Fourier basis.
    """
    n = N.bit_length()

    add = qk.QuantumRegister(n + 1, "add")
    flag = qk.QuantumRegister(1, "flag")
    qc = qk.QuantumCircuit(add, flag, name=f"phase_add_{k}_mod_{N}")

    # Add +k in the Fourier basis
    qc.compose(add_k_fourier(k, N), qubits=add, inplace=True)

    # Subtract N (by applying the inverse of the Fourier addition)
    qc.compose(add_k_fourier(N, N).inverse(), qubits=add, inplace=True)

    # Leave the Fourier basis so we can inspect whether the subtraction caused underflow
    qc.append(ql.QFTGate(n + 1).inverse(), add)

    # Copy the comparison bit into the flag ancilla.
    qc.cx(add[0], flag[0])

    # Return to Fourier basis
    qc.append(ql.QFTGate(n + 1), add)

    # If the subtraction underflowed, add N back.
    qc.compose(
        add_k_fourier(N, N).control(1),
        qubits=[flag[0], *add],
        inplace=True,
    )

    # Uncompute the flag ancilla.
    qc.compose(add_k_fourier(k, N).inverse(), qubits=add, inplace=True)
    qc.append(ql.QFTGate(n + 1).inverse(), add)

    # Flip flag if add[0] == 0
    qc.x(add[0])
    qc.cx(add[0], flag[0])
    qc.x(add[0])

    qc.append(ql.QFTGate(n + 1), add)
    qc.compose(add_k_fourier(k, N), qubits=add, inplace=True)

    return qc

In [4]:

def mul_out_k_mod(k: int, N: int) -> qk.QuantumCircuit:
    """
    Performs out-of-place modular multiplication where the second register is our accumulator register:
    
        |x>|0>  ->  |x>|k*x mod N>
    
    This is built from repeated controlled modular additions:
    
        k*x = sum_i x_i * (k * 2^i) mod N
    
    So for each bit x_i of the input register:
      if x_i == 1, add the corresponding shifted constant into the accumulator.

    Note that we need to use the accumulator register for this out-of-place multiplication because we
    can't directly implmenet in-place modular multipication in a quantum circuit, so we must use another
    register and then swap the bits in order to update our work register "x".
    
    Register layout in this flat circuit:
      [0 .. n-1]       = x register
      [n .. 2n]        = accumulator register (n+1 qubits total)
      [2n+1]           = flag ancilla
    """
    n = N.bit_length()
    total_qubits = 2 * n + 2

    qc = qk.QuantumCircuit(total_qubits, name=f"mul_out_{k}_mod_{N}")

    # Accumulator register used by the phase_adder:
    # in this layout, it is the n+1 qubits [n .. 2n] plus the last ancilla [2n+1]
    # We order them so they match the qubit order expected by phase_adder.
    acc_and_flag = list(range(n, 2 * n + 2))

    # Put the accumulator register into the Fourier basis
    qc.append(ql.QFTGate(n + 1), acc_and_flag[:-1])

    # For each bit of x, conditionally add the shifted constant
    for idx in range(n):
        # If x[idx] = 1, add k * 2^(n-1-idx) mod N into the accumulator
        shifted_const = (k * pow(2, n - 1 - idx, N)) % N

        qc.compose(
            phase_adder(shifted_const, N).control(1),
            qubits=[idx] + acc_and_flag,
            inplace=True,
        )

    # Return the accumulator to the computational basis
    qc.append(ql.QFTGate(n + 1).inverse(), acc_and_flag[:-1])

    return qc

In [5]:
def modular_multiply(k: int, N: int) -> qk.QuantumCircuit:
    """
    Performs in-place modular multiplication:

        |x> -> |k * x mod N>

    We do this by using the out-of-place modular multiplier, then swapping the
    result into the x register and uncomputing the old x via multiplication
    by k^{-1}.
    """
    n = N.bit_length()

    if math.gcd(k, N) != 1:
        raise ValueError("k must be invertible modulo N.")

    x = qk.QuantumRegister(n, "x")
    acc = qk.QuantumRegister(n + 1, "acc")
    flag = qk.QuantumRegister(1, "flag")

    qc = qk.QuantumCircuit(x, acc, flag, name=f"mod_mult_{k}_mod_{N}")

    # out-of-place multiplication
    qc.compose(mul_out_k_mod(k, N), inplace=True)

    # swap the computed value into the main x register
    for i in range(n):
        qc.swap(x[i], acc[i + 1])

    # uncompute the leftover old x using k^{-1}
    inv_k = pow(k, -1, N)
    qc.compose(mul_out_k_mod(inv_k, N).inverse(), inplace=True)

    return qc

### Building the Shor circuit

We now combine the modular multiplier with quantum phase estimation. The circuit initializes the work register $\ket{x} = \ket{1}$, applies Hadamards to the phase register, then for each phase qubit $i$ applies a controlled multiplication by $a^{2^i} \mod N$. An inverse QFT on the phase register followed by measurement yields an estimate of $s/r$, from which we extract the period $r$ classically using continued fractions.

In [6]:
def shor_period_finding_circuit(a: int, N: int, phase_bits: int | None = None) -> qk.QuantumCircuit:
    """
    This function creates the Shor circuit:
      1. Prepare phase register in uniform superposition
      2. Prepare work register in |1>
      3. For each phase qubit i, apply controlled multiplication by a^(2^i) mod N
      4. Apply inverse QFT to the phase register
      5. Measure the phase register

    Resulting measured phase bits should be approximately s/r where $r$ is the 
    """
    n = N.bit_length()
    t = phase_bits if phase_bits is not None else 2 * n

    phase = qk.QuantumRegister(t, "phase")
    x = qk.QuantumRegister(n, "x")
    acc = qk.AncillaRegister(n + 1, "acc")
    flag = qk.AncillaRegister(1, "flag")
    c_phase = qk.ClassicalRegister(t, "c_phase")

    qc = qk.QuantumCircuit(phase, x, acc, flag, c_phase, name=f"shor_a{a}_N{N}")

    # Initialize work register to |1>
    qc.x(x[n - 1])

    # Put phase register into uniform superposition
    qc.h(phase)

    # Controlled modular multiplications
    # For Shor, the i-th phase qubit controls multiplication by a^(2^i) mod N
    for i in range(t):
        multiplier = pow(a, 2**i, N)
        controlled_mult = modular_multiply(multiplier, N).control(1)

        qc.compose(
            controlled_mult,
            qubits=[phase[i], *x, *acc, *flag],
            inplace=True,
        )

    # Inverse QFT on the phase register
    qc.append(ql.QFTGate(t).inverse(), phase)

    # Measure only the phase register
    qc.measure(phase, c_phase)

    return qc

### Post-processing

We can run the circuit a large number of times to determine which measurements are the most common, then we store those in `counts`. What we're actually measuring in the phase bits is generally an approximation of $(s/r) 2^t$. To extract $r$ we calculate:

\begin{equation*}
    \frac{s}{r} \approx \frac{m}{2^t}
\end{equation*}

Where $m$ is our measured phase value from the circuit and $t$ is the number of bits we used for the phase register. Then, we use continued fractions to get the numerator ($s$) and denominator ($r$). Then we simply extract the denominator.

To find the factors of $N$ we calculate $g = \text{GCD}(a^{r/2} \pm 1, N)$ and we are likely to find a common divisor $g$, then the other factor is just $N / g$. This completes Shor's algorithm.

In [7]:
def extract_factors(a: int, N: int, counts: dict[str, int], phase_bits: int) -> tuple[int, int] | None:
    """
    Takes the measurement outcomes from running the circuit for $a$ and $N$ with $phase_bits$ number of bits
    used for phase and does the following:
        1. Utilizes continued-fraction expansion to find the period r from the circuit measurements
        2. Calculates gcd(a^(r/2) +- 1, N) to find factors of N
    """
    # Sort by the most-measured outcome
    sorted_outcomes = sorted(counts.items(), key=lambda x: x[1], reverse=True)

    for bitstring, _ in sorted_outcomes:
        measured = int(bitstring, 2)

        if measured == 0:
            continue

        # The measured value approximates s/r * 2^t where t is the number of phase bits,
        # so measured / 2^t ≈ s/r.
        phase_estimate = Fraction(measured, 2**phase_bits)

        # Use continued fractions to find r (the denominator).
        # limit_denominator(N) restricts to denominators that make sense for period-finding mod N.
        frac = phase_estimate.limit_denominator(N)
        r = frac.denominator

        print(f"  measured={measured}, phase≈{float(phase_estimate):.4f}, fraction={frac}, candidate r={r}")

        # Check that r is the order of a mod N (that is, a^r = 1 (mod N))
        if pow(a, r, N) != 1:
            continue

        # r must be even for factor extraction
        if r % 2 != 0:
            continue

        half_power = pow(a, r // 2, N)

        # Discard the trivial case a^(r/2) = -1 (mod N)
        if half_power == N - 1:
            continue

        p = math.gcd(half_power + 1, N)
        q = math.gcd(half_power - 1, N)

        # Determine if p or q are a factor
        for t in (p, q):
            if (1 < t < N) and (other := N // t) and (t * other == N):
                print(f"  -> Found factors: {t} x {other} = {N}")
                return (t, other)

    print("  No factors found from these measurements.")
    return None

In [8]:
def shor_factor(
    N: int,
    max_attempts: int = 5,
    phase_bits: int | None = None,
    shots: int = 1000,
) -> tuple[int, int] | None:
    """
    Helper function that runs the full Shor's algorithm for a given integer N. Will try up to max_attempts
    different values of $a$ that are co-prime to N. The circuit will use phase_bits number of qubits for the phase
    register, which defaults to 2 * n unless otherwise specified. The circuit runs through measurement $shots$ number
    of times, defaulting to 1,000.
    """
    if N < 4:
        raise ValueError(f"N must be >= 4, got {N}")

    # Trivial case: N is even
    if N % 2 == 0:
        print(f"{N} is even. Trivial factor: 2")
        return (2, N // 2)

    # Check if N is prime via trial division
    # note that we only need to check up to sqrt{N} values. If
    # we aren't able to find any divisors, we know this number is prime
    # and thus Shor's algorithm will fail
    for p in range(2, int(N**0.5) + 1):
        if N % p == 0:
            break
    else:
        raise ValueError(f"{N} is prime and cannot be factored.")

    # default to 2n bits for the phase register
    t = phase_bits if phase_bits is not None else 2 * N.bit_length()
    simulator = AerSimulator()

    for attempt in range(1, max_attempts + 1):
        # Pick random a, re-rolling if gcd(a, N) > 1 so the
        # quantum circuit always runs at least once.
        while True:
            a = random.randint(2, N - 1)
            g = math.gcd(a, N)
            if g > 1:
                print(f"  a={a} shares factor {g} with {N}, picking another a...")
                continue
            break

        print(f"Attempt {attempt}: a={a}")

        # Build, transpile, and simulate
        qc = shor_period_finding_circuit(a=a, N=N, phase_bits=phase_bits)
        qc_transpiled = transpile(qc, simulator)
        print(f"Pre-transpiled circuit utilizes {qc.size()} gates. Transpiled circuit utilizes {qc_transpiled.size()} gates")

        result = simulator.run(qc_transpiled, shots=shots).result()
        counts = result.get_counts()

        factors = extract_factors(a, N, counts, t)
        if factors is not None:
            return factors

        print(f"  Attempt {attempt} failed, retrying...\n")

    print(f"Failed to factor {N} after {max_attempts} attempts.")
    return None

In [9]:
N = 15
result = shor_factor(N)

if result:
    print(f"{N} = {result[0]} x {result[1]}")
else:
    print(f"No result found for N = {N}")


Attempt 1: a=11
Pre-transpiled circuit utilizes 26 gates. Transpiled circuit utilizes 205129 gates
  measured=128, phase≈0.5000, fraction=1/2, candidate r=2
  -> Found factors: 3 x 5 = 15
15 = 3 x 5
